In [1]:
from omegaconf import OmegaConf
import argparse
from models import get_model
from data import (
    apply_scaler,
    inverse_scaler,
    reshape_input_dims,
    get_validation_set,
    get_dataset,
    get_dataloaders,
)
from data.registry import load_train_test_set
from optimizers import get_optimizer
from losses import get_loss
from schedulers import get_scheduler
from metrics import get_metrics
from callbacks import get_callbacks
from trainers.base_trainer import BaseTrainer
from loggers import setup_logger, get_output_logger
from utils.loggers import setup_logger
from utils.seed import seed_everything
from utils.get_experiment_id import get_experiment_id
from utils.load_checkpoint import load_checkpoint
from utils.wandb_login import wandb_login
from utils.wandb_init import wandb_init
from utils.plots import (
    plot_inverse_transformed_outputs_and_mae,
    plot_average_mae_per_step,
)
import torch
import wandb
import os
import matplotlib.pyplot as plt
import numpy as np

In [2]:
    print("--------------   GPU  --------------------")
    print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES')}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Utilizando device: {device}")

--------------   GPU  --------------------
CUDA_VISIBLE_DEVICES: None
Utilizando device: cpu


In [3]:
# Este notebook tiene como objetivo cargar modelos desde checkpoint y evaluarlos sobre test.
# Alex nos ha pedido analizar el comportamiento del modelo en los peores casos.

dataset_1c_path = "108-24h-24h-3L-1c.pt"
dataset_2c_path = "108-24h-24h-3L-2c.pt"
dataset_path = [dataset_1c_path, dataset_2c_path]

model_1c_path = "108-24h-24h-3L-1c_cnn_lstm_8879af9f"
model_2c_path = "108-24h-24h-3L-2c_cnn_lstm_80a91774"
model_path = [model_1c_path, model_2c_path]

print(f"checkpoints/{model_1c_path}")
print(f"checkpoints/{model_2c_path}")

checkpoints/108-24h-24h-3L-1c_cnn_lstm_8879af9f
checkpoints/108-24h-24h-3L-2c_cnn_lstm_80a91774


In [4]:
for i in range(2):
    print(f"Evaluación dataset: {dataset_path[i]}, modelo: {model_path[i]}")

    config_path = f"checkpoints/{model_path[i]}/config.yaml"
    config = OmegaConf.load(config_path)
    print(config)

    train_set, test_set = load_train_test_set(config)
    input_size, output_size = train_set[0][0].shape, train_set[0][1].shape[0]
    print(f"Input size: {input_size}, Output size: {output_size}")

    # si los datos de entrada son multidimensionales, los invierte de (n, len) a (len, n) para que la LSTM los entienda
    train_set, test_set = reshape_input_dims(train_set, test_set)

    train_set_scaled, test_set_scaled, scaler = apply_scaler(
        train_set, test_set, config, debug=True
    )  # Aplica el escalador si es necesario

    train_splits, val_splits = get_validation_set(
        train_set_scaled, config
    )  # Devuelve una lista. Si es sin kfold solo un elemento en train y otro en val. Si es k-fold devuelve los k sets.

    folds_val_metrics, folds_test_metrics = {}, {}


    train_ds, val_ds, test_ds = get_dataset(
        train_splits[0], val_splits[0], test_set_scaled, config
    )
    train_loader, val_loader, test_loader = get_dataloaders(
        train_ds, val_ds, test_ds, config
    )
    # print dimensions of a minibatch
    print(
        f"Dimensiones de un minibatch: ({train_loader.batch_size}, {train_loader.dataset[0][0].shape}, {train_loader.dataset[0][1].shape})"
    )
    
    model = get_model(input_size, output_size, config.model).to(
        config.training.device
    )
    criterion = get_loss(config.loss)
    optimizer = get_optimizer(config.optimizer, model.parameters())
    scheduler = get_scheduler(config.scheduler, optimizer)
    callbacks = get_callbacks(config.callbacks)
    metrics = get_metrics(config.metrics)

    trainer = BaseTrainer(
            model,
            criterion,
            optimizer,
            scheduler,
            config,
            _,
            _,
            callbacks,
            metrics,
        )

    all_outputs = []
    for fold in range(5):
        print(f"k-model fold: {fold}")
        load_checkpoint(model, config, fold)
    
        test_metrics, inputs, outputs, targets = trainer.run_epoch(
            test_loader, mode="Test", return_preds=True
        )
    
        # Desescalamos las predicciones y los targets
        outputs = inverse_scaler(outputs, scaler)
        targets = inverse_scaler(targets, scaler)

        all_outputs.append(outputs)
    
    # TO DO: Hacer la media de los 5 outputs por cada par
    # all_outputs es una lista de 5 tensores (folds), cada uno con forma [N, 24]
    # Los apilamos en un tensor de forma [5, N, 24] y luego hacemos la media en dim=0
    outputs = [
    torch.stack(tensors, dim=0).mean(dim=0)   # tensors es una tupla de 5 tensores [24]
    for tensors in zip(*all_outputs)
    ]

    # Calcular MAE de cada par (hecha la media de los outputs de los 5 modelos)
    maes = []
    for i, (out, tgt) in enumerate(zip(outputs, targets)):
        mae = torch.mean(torch.abs(out - tgt)).item()
        maes.append((i, mae))
    
    # Ordenar por MAE
    maes_sorted = sorted(maes, key=lambda x: x[1])
    
    # Obtener índices del menor y mayor MAE
    idx_min, mae_min = maes_sorted[0]
    idx_max, mae_max = maes_sorted[-1]
    
    print(f"Menor MAE: idx test: {idx_min}, MAE par={mae_min:.4f}")
    print(f"Mayor MAE: idx test: {idx_max}, MAE par={mae_max:.4f}")
    # Función para graficar un par
    def plot_pair(idx, out, tgt, mae, title_extra=""):
        plt.figure(figsize=(8,4))
        plt.plot(out.numpy(), label="Output", marker="o")
        plt.plot(tgt.numpy(), label="Target", marker="x")
        plt.title(f"Par {idx} - MAE={mae:.4f} {title_extra}")
        plt.xlabel("Step")
        plt.ylabel("Valor")
        plt.legend()
        plt.grid(True)
        plt.show()
    
    # Graficar menor y mayor MAE
    plot_pair(idx_min, outputs[idx_min], targets[idx_min], mae_min, "(menor MAE)")
    plot_pair(idx_max, outputs[idx_max], targets[idx_max], mae_max, "(mayor MAE)")

    # Extraer solo los valores de MAE (sin los índices)
    mae_values = [m[1] for m in maes]
    
    # Histograma
    plt.figure(figsize=(8,5))
    plt.hist(mae_values, bins=20, edgecolor="black", alpha=0.7)
    plt.title("Distribución de MAEs por par")
    plt.xlabel("MAE")
    plt.ylabel("Frecuencia")
    plt.grid(True, linestyle="--", alpha=0.6)
    
    # Línea vertical con la media
    mae_mean = np.mean(mae_values)
    plt.axvline(mae_mean, color="red", linestyle="--", linewidth=2, label=f"Media = {mae_mean:.4f}")
    
    plt.legend()
    plt.show()

Evaluación dataset: 108-24h-24h-3L-1c.pt, modelo: 108-24h-24h-3L-1c_cnn_lstm_8879af9f
{'dataset': {'dataset_module': 'sacyr', 'root': './data/datasets', 'num_workers': 4, 'mode': 'reggresion', 'kfold': {'num_folds': 5, 'shuffle': True}, 'name': '108-24h-24h-3L-1c'}, 'scaler': {'name': 'standard'}, 'loss': {'name': 'mse'}, 'optimizer': {'name': 'adam', 'lr': 0.001, 'weight_decay': 0.0001, 'scheduler': 'step', 'step_size': 10, 'gamma': 0.1}, 'training': {'epochs': 100, 'batch_size': 1024, 'num_workers': 2, 'prefetch_factor': 3, 'seed': 42, 'device': 'cuda', 'log_dir': 'logs'}, 'scheduler': {'name': 'reduce_on_plateau', 'patience': 5, 'factor': 0.1}, 'metrics': [{'name': 'mae-sacyr-1h', 'prediction_horizon': 1}], 'callbacks': [{'name': 'checkpoint', 'dirpath': 'checkpoints/', 'monitor': 'Val_loss', 'mode': 'min'}, {'name': 'wandb_logger', 'project': 'sacyr', 'entity': 'inaki'}, {'name': 'early_stopping', 'patience': 15, 'monitor': 'Val_loss'}], 'wandb': {'project': 'sacyr', 'entity': 'ina

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3607113742925706 and num_layers=1
  warnings.warn(


RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx